# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id and name
print("Available Record Sets (referenced with @id):")
for record_set in dataset.record_sets:
    print(f"- {record_set.id}: {record_set.name}")

# For demonstration, pick one record set (the first one)
main_record_set = dataset.record_sets[0]
main_record_set_id = main_record_set.id
print(f"\nFields in the Record Set '{main_record_set_id}':")
for field in main_record_set.fields:
    print(f"  - {field.id}: {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into pandas DataFrames
record_sets_ids = [rec_set.id for rec_set in dataset.record_sets]

dataframes = {}

for rec_id in record_sets_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df

# Show columns for the main record set
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data, and grouping by a key attribute from the dataset.

In [ ]:
# Select numeric and group fields using @id
# These @id values should be adapted based on actual fields listed above
# Example field ids (replace as needed):
#   numeric_field_id = '@accession.coverage_percentage'
#   group_field_id = '@description.sample_type'

df = dataframes[main_record_set_id]
# Guess likely numeric field ids: try 'coverage_percentage', 'MW', 'peptide_count' from field names
# Find numeric fields
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in ['i', 'f']]
if not numeric_candidates:
    # Try to infer float columns even if initially all are object dtype
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in ['i', 'f']]

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    print("No numeric fields found.")
    numeric_field_id = None

# Choose a group field if available
possible_group_fields = [col for col in df.columns if 'sample' in col.lower() or 'type' in col.lower()]
group_field_id = possible_group_fields[0] if possible_group_fields else None

if numeric_field_id:
    threshold = np.nanpercentile(df[numeric_field_id], 75)  # 75th percentile as demo threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (showing group means):")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Histogram of the main numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group (if group field exists)
if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, extracting, and exploring the FAIR² mass spectrometry dataset using `mlcroissant`. We listed record sets and fields (by `@id`), loaded records, conducted basic filtering and normalization, and visualized the distribution and groupings within the dataset.

Further steps could include deeper statistical analysis, advanced visualizations, or integration with other protein/proteomics knowledge sources using the Croissant identifiers.
